In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import multiprocessing as mp

# 1. Path Resolution
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/pavel/academia/wintermute/mf-temporary')

# 2. The Great MPI Blindfold
keys_to_delete = [k for k in os.environ if k.startswith('SLURM') or k.startswith('OMPI_') or k.startswith('PRTE_') or 'HYDRA' in k]
for key in keys_to_delete:
    del os.environ[key]

# Prevent CPU hogging by the C++ backend
os.environ["OMP_NUM_THREADS"] = "1" 

# 3. Safe Multiprocessing
try:
    mp.set_start_method('spawn', force=True)
    print("Multiprocessing context set to:", mp.get_start_method())
except RuntimeError:
    pass

print(f"Loaded paths securely. MPI blindfolded. Ready for testing!")

In [ ]:
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from codes.neuron_simulation.config import NeuronSimulationConfig
from codes.network_params.loader import load_network_parameters
from codes.plotting import fig_plots
import codes.neuron_simulation as ns

from pydantic import BaseModel

project_path = repo_path / "projects" / "00_drafts_tests"
os.chdir(project_path)

class TestNeuronSimulationParams(BaseModel):
    neuron_simulation: NeuronSimulationConfig



In [ ]:
network_params = load_network_parameters(project_path / "params" / "network_params_new.yaml")
simulators = ["zerlaut2018", "pynn.nest"]


# Testing of `neuron_simulation` module

I have made big changes in neuron_simulation
- I have added new way of handling config parameters (using pydantic)
- I have split `__init__.py` into three files
    - `__init__.py` - importing and handling backend simulators and providing `run_neuron_simulation_workflow` function 
    - `__base__.py` - defines an abstract class of simulator, which should be the main implementation of each simulator backend
    - `pynn_simulator.py` previous simulation code with extra improvements

### Testing linear grid

In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": None,
        "grid": {
            "exc_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 10],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 10],
            },
            "inh_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 10],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 10]
            }
        },
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

neuron_results_list = []

for simulator in simulators:
    print(f"\nTesting simulator: {simulator}")
    test_workflow_params["neuron_simulation"]["simulator"] = simulator
    test_config = TestNeuronSimulationParams(**test_workflow_params).neuron_simulation

    print("Config loaded successfully!")
    print(f"Running for {test_config.simulation_time} ms")

    neuron_results_list.append(ns.run_neuron_simulation_workflow(test_config, network_params))

In [ ]:
for simulator, neuron_results in zip(simulators, neuron_results_list):
    fig_name = f"neuron_activity_test_linear_grid_{simulator.replace('.', '_')}.png"
    fig_path = project_path / "imgs" / fig_name
    fig_plots.fig_neuron_activity(
        neuron_results, 
        common_params={}, 
        fig_params={
            'tight_layout': True,
            'savefig': True,
            'savefig_path': fig_path,
            'title': f"Neuron Activity - {simulator}"
        }
    )
    display(Image(filename=str(fig_path)))

    fig_name = f"std_neuron_activity_test_linear_grid_{simulator.replace('.', '_')}.png"
    fig_path = project_path / "imgs" / fig_name
    fig_plots.fig_tf_fits_together(
        neuron_results, 
        {"exc_neuron": [],
        "inh_neuron": []
        }, 
        common_params={
            'labels' : [],
            'linestyles' : [],
            # 'xlim' : (0,7),
            'ylim' : (0, 60),
        }, 
        fig_params={
            'fontsize': 14,
            'figsize': (15, 10),  # width, height
            'tight_layout': True,
            'savefig': True,
            'savefig_path': fig_path,
            'title': f"Neuron Activity - {simulator} (STD)"
        })
    display(Image(filename=str(fig_path)))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for simulator, neuron_results, marker in zip(simulators, neuron_results_list, ['o', 'x']):
    axes[0].set_prop_cycle(None)
    axes[1].set_prop_cycle(None)    
    lines_exc = axes[0].plot(neuron_results['exc_neuron'].exc_rate_grid(), 
                 neuron_results['exc_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)

    lines_inh = axes[1].plot(neuron_results['inh_neuron'].exc_rate_grid(), 
                 neuron_results['inh_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)
    lines_exc[0].set_label(simulator)
    lines_inh[0].set_label(simulator)
axes[0].set_title("Excitatory Neuron")
axes[1].set_title("Inhibitory Neuron")
axes[0].set_xlabel("Excitatory Input Rate (Hz)")
axes[1].set_xlabel("Excitatory Input Rate (Hz)")
axes[0].set_ylabel("Output Rate (Hz)")
axes[1].set_ylabel("Output Rate (Hz)")
axes[0].legend()
axes[1].legend()
fig.suptitle("Comparison of Neuron Activity Across Simulators")
fig.tight_layout()


### Testing adaptive grid
Zerlaut is supper slow in here! (possibly due to no multiprocessing)

In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": None,
        "grid": {
            "exc_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 10],
                "out_rate_grid": [0, 30, 10],
                "n_coarse_interpolation_points" : 16, 
            },
            "inh_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 10],
                "out_rate_grid": [0, 30, 10],
                "n_coarse_interpolation_points" : 16, 
            }
        },
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

neuron_results_list = []

for simulator in simulators:
    print(f"\nTesting simulator: {simulator}")
    test_workflow_params["neuron_simulation"]["simulator"] = simulator
    test_config = TestNeuronSimulationParams(**test_workflow_params).neuron_simulation

    print("Config loaded successfully!")
    print(f"Running for {test_config.simulation_time} ms")

    neuron_results_list.append(ns.run_neuron_simulation_workflow(test_config, network_params))

In [ ]:
for simulator, neuron_results in zip(simulators, neuron_results_list):
    fig_name = f"neuron_activity_test_adaptive_grid_{simulator.replace('.', '_')}.png"
    fig_path = project_path / "imgs" / fig_name
    fig_plots.fig_neuron_activity(
        neuron_results, 
        common_params={}, 
        fig_params={
            'tight_layout': True,
            'savefig': True,
            'savefig_path': fig_path,
            'title': f"Neuron Activity - {simulator}"
        }
    )
    display(Image(filename=str(fig_path)))

    fig_name = f"std_neuron_activity_test_adaptive_grid_{simulator.replace('.', '_')}.png"
    fig_path = project_path / "imgs" / fig_name
    fig_plots.fig_tf_fits_together(
        neuron_results, 
        {"exc_neuron": [],
        "inh_neuron": []
        }, 
        common_params={
            'labels' : [],
            'linestyles' : [],
            # 'xlim' : (0,7),
            'ylim' : (0, 60),
        }, 
        fig_params={
            'fontsize': 14,
            'figsize': (15, 10),  # width, height
            'tight_layout': True,
            'savefig': True,
            'savefig_path': fig_path,
            'title': f"Neuron Activity - {simulator} (STD)"
        })
    display(Image(filename=str(fig_path)))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for simulator, neuron_results, marker in zip(simulators, neuron_results_list, ['o', 'x']):
    axes[0].set_prop_cycle(None)
    axes[1].set_prop_cycle(None)    
    lines_exc = axes[0].plot(neuron_results['exc_neuron'].exc_rate_grid(), 
                 neuron_results['exc_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)
    
    lines_inh = axes[1].plot(neuron_results['inh_neuron'].exc_rate_grid(), 
                 neuron_results['inh_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)
    lines_exc[0].set_label(simulator)
    lines_inh[0].set_label(simulator)
axes[0].set_title("Excitatory Neuron")
axes[1].set_title("Inhibitory Neuron")
axes[0].set_xlabel("Excitatory Input Rate (Hz)")
axes[1].set_xlabel("Excitatory Input Rate (Hz)")
axes[0].set_ylabel("Output Rate (Hz)")
axes[1].set_ylabel("Output Rate (Hz)")
axes[0].legend()
axes[1].legend()
fig.suptitle("Comparison of Neuron Activity Across Simulators")
fig.tight_layout()

### Testing custom grid

In [ ]:
data_path = project_path.parent / "01_fitting_tf" / "zerlaut2018_data"
custom_grid = {
    "exc_neuron": {},
    "inh_neuron": {}
}

for neuron_name, cell_type in zip(["exc_neuron", "inh_neuron"], ["RS", "FS"]):
    data = np.load(data_path / f"{cell_type}-cell_CONFIG1.npy", allow_pickle=True)
    custom_grid[neuron_name] = {
        "grid_type": "custom",
        "exc_rate_grid": data[2].T,
        "inh_rate_grid": np.stack([data[3]]*10, axis=1).T,
    }


In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": custom_grid,
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

neuron_results_list = []

for simulator in simulators:
    print(f"\nTesting simulator: {simulator}")
    test_workflow_params["neuron_simulation"]["simulator"] = simulator
    test_config = TestNeuronSimulationParams(**test_workflow_params).neuron_simulation

    print("Config loaded successfully!")
    print(f"Running for {test_config.simulation_time} ms")

    neuron_results_list.append(ns.run_neuron_simulation_workflow(test_config, network_params))

In [ ]:
for simulator, neuron_results in zip(simulators, neuron_results_list):
    fig_name = f"neuron_activity_test_custom_grid_{simulator.replace('.', '_')}.png"
    fig_path = project_path / "imgs" / fig_name
    fig_plots.fig_neuron_activity(
        neuron_results, 
        common_params={}, 
        fig_params={
            'tight_layout': True,
            'savefig': True,
            'savefig_path': fig_path,
            'title': f"Neuron Activity - {simulator}"
        }
    )
    display(Image(filename=str(fig_path)))

    fig_name = f"std_neuron_activity_test_custom_grid_{simulator.replace('.', '_')}.png"
    fig_path = project_path / "imgs" / fig_name
    fig_plots.fig_tf_fits_together(
        neuron_results, 
        {"exc_neuron": [],
        "inh_neuron": []
        }, 
        common_params={
            'labels' : [],
            'linestyles' : [],
            # 'xlim' : (0,7),
            'ylim' : (0, 60),
        }, 
        fig_params={
            'fontsize': 14,
            'figsize': (15, 10),  # width, height
            'tight_layout': True,
            'savefig': True,
            'savefig_path': fig_path,
            'title': f"Neuron Activity - {simulator} (STD)"
        })
    display(Image(filename=str(fig_path)))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for simulator, neuron_results, marker in zip(simulators, neuron_results_list, ['o', 'x']):
    axes[0].set_prop_cycle(None)
    axes[1].set_prop_cycle(None)    
    lines_exc = axes[0].plot(neuron_results['exc_neuron'].exc_rate_grid(), 
                 neuron_results['exc_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)
    
    lines_inh = axes[1].plot(neuron_results['inh_neuron'].exc_rate_grid(), 
                 neuron_results['inh_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)
    lines_exc[0].set_label(simulator)
    lines_inh[0].set_label(simulator)
axes[0].set_title("Excitatory Neuron")
axes[1].set_title("Inhibitory Neuron")
axes[0].set_xlabel("Excitatory Input Rate (Hz)")
axes[1].set_xlabel("Excitatory Input Rate (Hz)")
axes[0].set_ylabel("Output Rate (Hz)")
axes[1].set_ylabel("Output Rate (Hz)")
axes[0].legend()
axes[1].legend()
fig.suptitle("Comparison of Neuron Activity Across Simulators")
fig.tight_layout()

### Testing cpu=1

In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 5],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 5],
            },
            "inh_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 5],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 5]
            }
        },
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 1
    }
}

test_config = TestNeuronSimulationParams(**test_workflow_params).neuron_simulation

print("Config loaded successfully!")
print(f"Running for {test_config.simulation_time} ms")

neuron_results = ns.run_neuron_simulation_workflow(test_config, network_params)

In [ ]:
fig_name = "neuron_activity_cpu1_test_linear_grid.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    }
)
display(Image(filename=str(fig_path)))

fig_name = "std_neuron_activity_cpu1_test_linear_grid.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
     "inh_neuron": []
     }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 60),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    })
display(Image(filename=str(fig_path)))

### Consistency test
**Goal**: Ensure that simulate_adex_batch_multiprocess produces exactly the same statistical results as simulate_adex_batch when given the same seeds.

**Verification**: Compare the mean firing rates and membrane potentials between the two methods for a small 2×2 grid.

In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 5],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 5],
            },
            "inh_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 5],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 5]
            }
        },
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 1
    }
}

test_config = TestNeuronSimulationParams(**test_workflow_params).neuron_simulation

print("Config loaded successfully!")
print(f"Running for {test_config.simulation_time} ms")

neuron_results = ns.run_neuron_simulation_workflow(test_config, network_params)

test_config.cpus = 16
neuron_results_multiprocess = ns.run_neuron_simulation_workflow(test_config, network_params)


In [ ]:
for neuron_name in network_params.internal_neurons:
    results = neuron_results[neuron_name]
    results_mp = neuron_results_multiprocess[neuron_name]
    for attr in ['out_rate_mean', 'out_rate_std', 'adaptation_mean', 'adaptation_std', 'voltage_mean', 'voltage_std', 'exc_conductance_mean', 'inh_conductance_mean', 'exc_conductance_std', 'inh_conductance_std']:
        val = getattr(results, attr)
        val_mp = getattr(results_mp, attr)
        assert np.array_equal(val, val_mp), f"Mismatch in {attr} for {neuron_name}"

print("All attributes match between single-process and multi-process runs. No state or seed leakage detected.")


### Test generation of config params

`python -m codes.controller.config --template`

`python -m codes.controller.config --schema`


In [ ]:
!python -m codes.controller.config --template

In [ ]:
!python -m codes.controller.config --schema

### Other tests
- `execution_mode = "load"`
- test if it fails properly, (test the inputs such, give wrong numbers, values etc.)
- test optional values
- test config generation

### 1. The "Adaptive Grid" Stress Test

Since this involves Scipy interpolation and dynamic boundaries, it is the most likely place for edge-case bugs.
- The "Zero Firing" Edge Case: Does the inverse interpolation correctly fall back to 0.0 if the coarse grid simulation never fires? (Watch the console output to ensure it doesn't crash on inh_rate = 0 or very high inh_rate).
- Boundary Clipping: Verify that out_rate_targets that exceed the maximum firing rate of the coarse grid are safely clipped to the maximum possible input.

### 2. The "Custom Grid" Validation Test

This should be quick but is vital for user safety.
- Shape Mismatch: Pass a custom exc_rate_grid of shape (10, 10) and an inh_rate_grid of shape (10, 15) and verify that your ValueError correctly catches it.
- Happy Path: Pass a valid custom 2D grid and ensure the simulator iterates over it without complaining.
In computational physics, parallelized code often introduces race conditions that destroy determinism. We need to prove our seed logic is airtight.